# Solutions — First-Class Functions & Closures

Exercise notebook: `05_functions_and_oop/02_first_class_functions_closures.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Independent counters

**Working solution**

In [ ]:
def make_sequence(prefix, start=1, width=6):
    current = start - 1
    def next_id():
        nonlocal current
        current += 1
        return f"{prefix}{current:0{width}d}"
    return next_id

**Explanation**

`nonlocal` lets the inner function rebind `current` in the enclosing scope. Each call to `make_sequence` creates a new scope → independent state.

**Complexity:** O(1).

**Common mistakes**

- Forgetting `nonlocal` → `UnboundLocalError`.
- Using a global counter shared by all sequences.

## Exercise 2 — 🟡 Medium · Configured validators via closures

**Working solution**

In [ ]:
def required(field):
    def validate(record):
        value = record.get(field)
        return f"{field} is required" if value is None or str(value).strip() == "" else None
    return validate

def max_length(field, n):
    def validate(record):
        value = record.get(field)
        return f"{field} longer than {n}" if value is not None and len(str(value)) > n else None
    return validate

def one_of(field, *allowed):
    def validate(record):
        value = record.get(field)
        return f"{field} must be one of {', '.join(map(str, allowed))}" if value is not None and value not in allowed else None
    return validate

def check(record, *validators):
    return [msg for v in validators if (msg := v(record)) is not None]

**Explanation**

Each factory captures its configuration in a closure; the result is just a function, composable in a list. Compare to a `Validator` class hierarchy — same power, a fraction of the code.

**Complexity:** O(validators).

**Common mistakes**

- Late binding: creating closures in a loop (`[lambda r: r[f] for f in fields]`) all capture the *last* `f`. Bind via a factory or default arg.

## Exercise 3 — 🟡 Medium · Memoize with an inspectable cache

**Working solution**

In [ ]:
import functools

def memoize(func):
    cache = {}

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        try:
            hash(key)
        except TypeError:
            wrapper.calls += 1
            return func(*args, **kwargs)
        if key not in cache:
            wrapper.calls += 1
            cache[key] = func(*args, **kwargs)
        return cache[key]

    wrapper.cache = cache
    wrapper.calls = 0
    return wrapper

**Explanation**

Functions are objects, so you can attach attributes to them (`wrapper.cache`). `functools.wraps` copies the name/docstring so tools and logs
show the original function. In practice use `functools.lru_cache`/`cache` — but now you know what they do.

**Complexity:** O(1) average lookup.

**Common mistakes**

- Keying only on `args` → `f(1, x=2)` and `f(1, x=3)` collide.
- Caching `None` results incorrectly with `if cache.get(key):`.

## Exercise 4 — 🟠 Hard · Sliding-window rate limiter

**Working solution**

In [ ]:
import time
from collections import deque

def make_rate_limiter(max_calls, period, clock=time.monotonic):
    calls = deque()

    def _evict(now):
        while calls and calls[0] <= now - period:
            calls.popleft()

    def allow():
        now = clock()
        _evict(now)
        if len(calls) < max_calls:
            calls.append(now)
            return True
        return False

    def remaining():
        _evict(clock())
        return max_calls - len(calls)

    allow.remaining = remaining
    return allow

**Explanation**

Injecting `clock` (a function!) makes time-dependent code deterministic in tests — the functional equivalent of injecting a `Clock` in Java.
Two closures share the same `calls` deque.

**Complexity:** O(1) amortized.

**Common mistakes**

- Calling `time.time()` directly → tests need `sleep`.
- Counting rejected calls toward the limit.

## Challenge — 🔴 Very Hard · An event bus from closures

**Working solution**

In [ ]:
def make_event_bus():
    handlers = {}

    def subscribe(event_type, handler):
        handlers.setdefault(event_type, []).append(handler)
        def unsubscribe():
            bucket = handlers.get(event_type, [])
            if handler in bucket:
                bucket.remove(handler)
        return unsubscribe

    def publish(event_type, payload):
        targets = list(handlers.get(event_type, [])) + (list(handlers.get("*", [])) if event_type != "*" else [])
        delivered, errors = 0, []
        for handler in targets:
            try:
                handler(event_type, payload)
                delivered += 1
            except Exception as exc:
                errors.append((getattr(handler, "__name__", repr(handler)), repr(exc)))
        return {"delivered": delivered, "errors": errors}

    return subscribe, publish

**Explanation**

The bus's state lives in one closure scope shared by `subscribe`, `publish` and every `unsubscribe`. Copying the handler list before iterating
(`list(...)`) is the snapshot that makes mutation during iteration safe.

**Complexity:** O(handlers) per publish.

**Common mistakes**

- Iterating the live list while a handler unsubscribes itself → skipped handlers.
- Letting one exception abort delivery.